In [14]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY=os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT=os.getenv("LANGCHAIN_PROJECT")

In [16]:
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]=LANGCHAIN_PROJECT

In [17]:
from langchain_groq import ChatGroq


llm=ChatGroq(model="llama-3.1-8b-instant")

In [18]:
## basic chatbot

# while True:
#     question=input("type your question. if you want to quit the chat write quit")
#     if question !="quit":
#         print(llm.invoke(question).content)
#     else:
#         print("goodbye take care yourself")
#         break

In [19]:
## model with memory 

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import AIMessage

In [20]:
store={}


In [21]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [22]:
config = {"configurable": {"session_id": "firstchat"}}

In [23]:
model_with_memory=RunnableWithMessageHistory(llm,get_session_history)

In [24]:
model_with_memory.invoke(("Hi! I'm sanskar"),config=config).content

'Nice to meet you, Sanskar. Is there something I can help you with or would you like to chat?'

In [25]:
model_with_memory.invoke(("Hi! what is my name"),config=config).content

'Your name is Sanskar.'

In [26]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough , RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [27]:
## reading the text files from the directory and converting them to Document structures

loader = DirectoryLoader('./data', glob="./*.txt", loader_cls=TextLoader)
docs = loader.load()

print("Docs loaded:", len(docs))

## creating chunks of the Documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len
)
new_docs = text_splitter.split_documents(documents=docs)

doc_strings=[doc.page_content for doc in new_docs]

doc_strings

Docs loaded: 1


['Agentic AI is a type of artificial intelligence',
 'that goes beyond just responding to user inputs',
 'inputs and instead can independently plan, make',
 'make decisions, and take actions to achieve a',
 'achieve a goal. Unlike traditional or generative',
 'AI systems that simply generate outputs when',
 'when prompted, agentic AI systems operate in a',
 'in a loop of perceiving information, reasoning',
 'reasoning about it, executing tasks (like calling',
 'calling APIs or using tools), and adapting based',
 'based on results, allowing them to handle',
 'to handle complex, multi-step workflows with',
 'with minimal human intervention.']

In [28]:
## creating embeddings of the chunks
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


## storing the embeddings into the vector db
db = Chroma.from_documents(new_docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 4})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11922.54it/s]


In [29]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

In [30]:
retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
    )

In [31]:
question ="what is Agentic AI?"
print(retrieval_chain.invoke(question))

According to the context, Agentic AI is a type of artificial intelligence.


### Tools and Agents

In [57]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [58]:
api_wrapper=WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=100)

In [59]:
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)



In [60]:
wiki.name

'wikipedia'

In [61]:
wiki.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [62]:
print(wiki.run({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


In [63]:
from langchain.agents import create_agent

In [64]:
agent=create_agent(
    model=llm,
    tools=[wiki],
    system_prompt="You are a helpful assistant"
)

In [65]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is GDP of India?"}
    ]
})

In [66]:
print(response["messages"][-1].content)

I apologize but I was unable to verify the information about the GDP of India.
